In [1]:
import sys
import json
import csv
import yaml

import importlib

import math

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

from datetime import time
from datetime import date
from datetime import datetime

import pprint

import psycopg2
from sqlalchemy import create_engine, text as sql_text

from pymongo import MongoClient

In [2]:
db_username = os.environ['db_username']
db_password = os.environ['db_password']

db_eng = create_engine('postgresql+psycopg2://' + db_username + ':' + db_password + '@localhost:5432/Small_Examples',
                       connect_args={'options': '-csearch_path={}'.format('company_no_caps')},
                       isolation_level = 'SERIALIZABLE')

In [3]:
client = MongoClient()

In [29]:
# need this function if any date fields in tables will be loaded into mongodb
# In that case, you need to convert date type into datetime type
#    Note: if you have null values, that adds another wrinkle, and you
#          have to deal with df NaT values by converting them to None in 
#          a list of python dictionaries

def convert_date_to_datetime(dt):
    if pd.isnull(dt):           # tests whether dt is None, NaN, or DaT (not a date)
        return None
    # elif type(dt) == pd._libs.tslibs.nattype.NaTType:  # including this, but see below
    #     return None
    else:
        temp = datetime(dt.year, dt.month, dt.day)
        ts = temp.timestamp()
        new_dt = datetime.fromtimestamp(ts)
        return new_dt


In [30]:
query = "SELECT * FROM department"

with db_eng.connect() as conn:
    df_department = pd.read_sql(query, con=conn)

print(df_department)
print("\n")

# replacing all dates by datetimes (or None)
# NOTE: this works in this situation because there are no null values

df_department['mgr_start_date'] = df_department['mgr_start_date'].apply(convert_date_to_datetime)

department_records = df_department.to_dict(orient='records')

print(department_records)

db_company = client.company
db_company.department.drop()
department_mongo = db_company.department
department_mongo.insert_many(department_records)

print(db_company.list_collection_names())
print("\n")


for doc in department_mongo.find():
    print(doc)


            dname  dnumber    mgr_ssn mgr_start_date
0        Research        5  333445555     1988-05-22
1  Administration        4  987654321     1995-01-01
2    Headquarters        1  888665555     1981-06-19


[{'dname': 'Research', 'dnumber': 5, 'mgr_ssn': '333445555', 'mgr_start_date': Timestamp('1988-05-22 00:00:00')}, {'dname': 'Administration', 'dnumber': 4, 'mgr_ssn': '987654321', 'mgr_start_date': Timestamp('1995-01-01 00:00:00')}, {'dname': 'Headquarters', 'dnumber': 1, 'mgr_ssn': '888665555', 'mgr_start_date': Timestamp('1981-06-19 00:00:00')}]
['dependent', 'employee', 'dept_locations', 'project', 'department', 'works_on']


{'_id': ObjectId('683fd014cd226d66000cf4ba'), 'dname': 'Research', 'dnumber': 5, 'mgr_ssn': '333445555', 'mgr_start_date': datetime.datetime(1988, 5, 22, 0, 0)}
{'_id': ObjectId('683fd014cd226d66000cf4bb'), 'dname': 'Administration', 'dnumber': 4, 'mgr_ssn': '987654321', 'mgr_start_date': datetime.datetime(1995, 1, 1, 0, 0)}
{'_id': ObjectId('683fd014

['dependent', 'department', 'dept_locations', 'employee', 'works_on', 'project']


{'_id': ObjectId('683faf48cd226d66000cf48a'), 'dname': 'Research', 'dnumber': 5, 'mgr_ssn': '333445555', 'mgr_start_date': datetime.datetime(1988, 5, 22, 0, 0)}
{'_id': ObjectId('683faf48cd226d66000cf48b'), 'dname': 'Administration', 'dnumber': 4, 'mgr_ssn': '987654321', 'mgr_start_date': datetime.datetime(1995, 1, 1, 0, 0)}
{'_id': ObjectId('683faf48cd226d66000cf48c'), 'dname': 'Headquarters', 'dnumber': 1, 'mgr_ssn': '888665555', 'mgr_start_date': datetime.datetime(1981, 6, 19, 0, 0)}


In [16]:
query = "SELECT * FROM dependent"


with db_eng.connect() as conn:
    df_dependent = pd.read_sql(query, con=conn)

df_dependent['bdate'] = df_dependent['bdate'].apply(convert_date_to_datetime)
print(df_dependent)
print("\n")

dependent_records = df_dependent.to_dict(orient='records')

print(dependent_records)
print("\n")
    

db_company = client.company
db_company.dependent.drop()
dependent_mongo = db_company.dependent
dependent_mongo.insert_many(dependent_records)

print(db_company.list_collection_names())
print("\n")


for doc in dependent_mongo.find():
    print(doc)

<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
        essn dependent_name gender      bdate relationship
0  333445555          Alice      F 1986-04-05     Daughter
1  333445555       Theodore      M 1983-10-25          Son
2  333445555            Joy      F 1958-05-03       Spouse
3  987654321          Abner      M 1942-02-28       Spouse
4  123456789        Michael      M 1988-01-04          Son
5  123456789          Alice      F 1988-12-30     Daughter
6  123456789      Elizabeth      F 1967-05-05       Spouse


[{'essn': '333445555', 'dependent_name': 'Alice', 'gender': 'F', 'bdate': Timestamp('1986-04-05 00:00:00'), 'relationship': 'Daughter'}, {'essn': '333445555', 'dependent_name': 'Theodore', 'gender': 'M', 'bdate': Timestamp('1983-10-25 00:00:00'), 'relationship': 'Son'}, {'essn': '333445555', 'dependent_name': 'Joy', 'gender': 'F

In [17]:
query = "SELECT * FROM dept_locations"

with db_eng.connect() as conn:
    df_dept_locations = pd.read_sql(query, con=conn)

print(df_dept_locations)
print("\n")

dept_locations_records = df_dept_locations.to_dict(orient='records')

print(dept_locations_records)
print("\n")

db_company = client.company
db_company.dept_locations.drop()
dept_locations_mongo = db_company.dept_locations
dept_locations_mongo.insert_many(dept_locations_records)

print(db_company.list_collection_names())
print("\n")


for doc in dept_locations_mongo.find():
    print(doc)

   dnumber  dlocation
0        1    Houston
1        4   Stafford
2        5   Bellaire
3        5  Sugarland
4        5    Houston


[{'dnumber': 1, 'dlocation': 'Houston'}, {'dnumber': 4, 'dlocation': 'Stafford'}, {'dnumber': 5, 'dlocation': 'Bellaire'}, {'dnumber': 5, 'dlocation': 'Sugarland'}, {'dnumber': 5, 'dlocation': 'Houston'}]


['dependent', 'department', 'dept_locations', 'employee', 'works_on', 'project']


{'_id': ObjectId('683faf79cd226d66000cf494'), 'dnumber': 1, 'dlocation': 'Houston'}
{'_id': ObjectId('683faf79cd226d66000cf495'), 'dnumber': 4, 'dlocation': 'Stafford'}
{'_id': ObjectId('683faf79cd226d66000cf496'), 'dnumber': 5, 'dlocation': 'Bellaire'}
{'_id': ObjectId('683faf79cd226d66000cf497'), 'dnumber': 5, 'dlocation': 'Sugarland'}
{'_id': ObjectId('683faf79cd226d66000cf498'), 'dnumber': 5, 'dlocation': 'Houston'}


In [18]:
query = "SELECT * FROM employee"


with db_eng.connect() as conn:
    df_employee = pd.read_sql(query, con=conn)

df_employee['bdate'] = df_employee['bdate'].apply(convert_date_to_datetime)
print(df_employee)
print("\n")

employee_records = df_employee.to_dict(orient='records')

print(employee_records)
print("\n")
    

db_company = client.company
db_company.employee.drop()
employee_mongo = db_company.employee
employee_mongo.insert_many(employee_records)

print(db_company.list_collection_names())
print("\n")


for doc in employee_mongo.find():
    print(doc)

<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
<class 'datetime.datetime'>
      fname minit    lname        ssn      bdate                 address  \
0      John     B    Smith  123456789 1965-01-09  731-Fondren-Houston-TX   
1  Franklin     T     Wong  333445555 1955-12-08     638-Voss-Houston-TX   
2    Alicia     J   Zelaya  999887777 1968-01-19   3321-Castle-Spring-TX   
3  Jennifer     S  Wallace  987654321 1941-06-20   291-Berry-Bellaire-TX   
4    Ramesh     K  Narayan  666884444 1962-09-15  975-Fire-Oak-Humble-TX   
5     Joyce     A  English  453453453 1972-07-31    5631-Rice-Houston-TX   
6     Ahmad     V   Jabbar  987987987 1969-03-29   980-Dallas-Houston-TX   
7     James     E     Borg  888665555 1937-11-10    450-Stone-Houston-TX   

  gender  salary  super_ssn  dno  
0      M   30000  333445555    5  
1      M   40000  888

In [19]:
query = "SELECT * FROM project"

with db_eng.connect() as conn:
    df_project = pd.read_sql(query, con=conn)

print(df_project)
print("\n")

project_records = df_project.to_dict(orient='records')

print(project_records)
print("\n")

db_company = client.company
db_company.project.drop()
project_mongo = db_company.project
project_mongo.insert_many(project_records)

print(db_company.list_collection_names())
print("\n")


for doc in project_mongo.find():
    print(doc)

             pname  pnumber  plocation  dnum
0         ProductX        1   Bellaire     5
1         ProductY        2  Sugarland     5
2         ProductZ        3    Houston     5
3  Computerization       10   Stafford     4
4   Reorganization       20    Houston     1
5      Newbenefits       30   Stafford     4


[{'pname': 'ProductX', 'pnumber': 1, 'plocation': 'Bellaire', 'dnum': 5}, {'pname': 'ProductY', 'pnumber': 2, 'plocation': 'Sugarland', 'dnum': 5}, {'pname': 'ProductZ', 'pnumber': 3, 'plocation': 'Houston', 'dnum': 5}, {'pname': 'Computerization', 'pnumber': 10, 'plocation': 'Stafford', 'dnum': 4}, {'pname': 'Reorganization', 'pnumber': 20, 'plocation': 'Houston', 'dnum': 1}, {'pname': 'Newbenefits', 'pnumber': 30, 'plocation': 'Stafford', 'dnum': 4}]


['dependent', 'employee', 'department', 'dept_locations', 'project', 'works_on']


{'_id': ObjectId('683faf93cd226d66000cf4a1'), 'pname': 'ProductX', 'pnumber': 1, 'plocation': 'Bellaire', 'dnum': 5}
{'_id': ObjectId('683faf

In [20]:
query = "SELECT * FROM works_on"

with db_eng.connect() as conn:
    df_works_on = pd.read_sql(query, con=conn)

print(df_works_on)
print("\n")

works_on_records = df_works_on.to_dict(orient='records')

print(works_on_records)
print("\n")

db_company = client.company
db_company.works_on.drop()
works_on_mongo = db_company.works_on
works_on_mongo.insert_many(works_on_records)

print(db_company.list_collection_names())
print("\n")


for doc in works_on_mongo.find():
    print(doc)

         essn  pno  hours
0   123456789    1   32.5
1   123456789    2    7.5
2   666884444    3   40.0
3   453453453    1   20.0
4   453453453    2   20.0
5   333445555    2   10.0
6   333445555    3   10.0
7   333445555   10   10.0
8   333445555   20   10.0
9   999887777   30   30.0
10  999887777   10   10.0
11  987987987   10   35.0
12  987987987   30    5.0
13  987654321   30   20.0
14  987654321   20   15.0
15  888665555   20    NaN


[{'essn': '123456789', 'pno': 1, 'hours': 32.5}, {'essn': '123456789', 'pno': 2, 'hours': 7.5}, {'essn': '666884444', 'pno': 3, 'hours': 40.0}, {'essn': '453453453', 'pno': 1, 'hours': 20.0}, {'essn': '453453453', 'pno': 2, 'hours': 20.0}, {'essn': '333445555', 'pno': 2, 'hours': 10.0}, {'essn': '333445555', 'pno': 3, 'hours': 10.0}, {'essn': '333445555', 'pno': 10, 'hours': 10.0}, {'essn': '333445555', 'pno': 20, 'hours': 10.0}, {'essn': '999887777', 'pno': 30, 'hours': 30.0}, {'essn': '999887777', 'pno': 10, 'hours': 10.0}, {'essn': '987987987', 'p

###  <span style=color:blue>This (should be) a MongoDB equivalent to an SQL query that was on 2025 miderm </span>

###  <span style=color:blue>View this in raw text mode, not formatted markdown </span>

<span style=color:blue>The English version of the query was:
(9 points) Consider the Company example shown in Figures 3 and 4. Write an SQL query that gives, for each
project for which the average salary of employees working on the project is ≥ $28, 000:
• the project name and number,
• the name and number of the department that manages that project,
• the count of employees working on the project, and
• the average salary of all employees who work on the project.
(Recall that in this example, the "dnum" field of the PROJECT relation refers to the department that is
managing it.) The answer to the query should be sorted by project name 
</span>

<span style=color:blue>The SQL is:

SELECT p.pname, p.pnumber, d.dname, d.dnumber,
count(*) emp count, AVG(e.salary) avg salary
FROM project p, department d, works on w, employee e
WHERE d.dnumber = p.dnum
AND w.pno = p.pnumber
AND e.ssn = w.essn
GROUP BY p.pname, p.pnumber, d.dname, d.dnumber
HAVING AVG(e.salary) >= 28000
ORDER BY p.pname

</span>


In [97]:
db_company = client.company


pipeline = [
    # 1. Join with department
    {
        "$lookup": {
            "from": "department",
            "localField": "dnum",
            "foreignField": "dnumber",
            "as": "dept"
        }
    },
    {"$unwind": "$dept"},

    # 2. Join with works_on
    {
        "$lookup": {
            "from": "works_on",
            "localField": "pnumber",
            "foreignField": "pno",
            "as": "works"
        }
    },
    {"$unwind": "$works"},

    # 3. Join with employee
    {
        "$lookup": {
            "from": "employee",
            "localField": "works.essn",
            "foreignField": "ssn",
            "as": "emp"
        }
    },
    {"$unwind": "$emp"},

    # 4. Group by pname, pnumber, dname, dnumber
    {
        "$group": {
            "_id": {
                "pname": "$pname",
                "pnumber": "$pnumber",
                "dname": "$dept.dname",
                "dnumber": "$dept.dnumber"
            },
            "emp_count": {"$sum": 1},
            "avg_salary": {"$avg": "$emp.salary"}
        }
    },

    # 5. Filter: HAVING avg_salary >= 28000
    {"$match": {"avg_salary": {"$gte": 28000}}},

    # 6. Sort by pname
    {"$sort": {"_id.pname": 1}},

    # 7. Optional: reshape output
    {
        "$project": {
            "_id": 0,
            "pname": "$_id.pname",
            "pnumber": "$_id.pnumber",
            "dname": "$_id.dname",
            "dnumber": "$_id.dnumber",
            "emp_count": 1,
            "avg_salary": 1
        }
    }
]


result = db_company.project.aggregate(pipeline)

# Print result
for doc in result:
    ordered = {
        "pname": doc["pname"],
        "pnumber": doc["pnumber"],
        "dname": doc["dname"],
        "dnumber": doc["dnumber"],
        "emp_count": doc["emp_count"],
        "avg_salary": doc["avg_salary"]
    }
    print(ordered)

{'pname': 'Computerization', 'pnumber': 10, 'dname': 'Administration', 'dnumber': 4, 'emp_count': 3, 'avg_salary': 30000.0}
{'pname': 'Newbenefits', 'pnumber': 30, 'dname': 'Administration', 'dnumber': 4, 'emp_count': 3, 'avg_salary': 31000.0}
{'pname': 'ProductY', 'pnumber': 2, 'dname': 'Research', 'dnumber': 5, 'emp_count': 3, 'avg_salary': 31666.666666666668}
{'pname': 'ProductZ', 'pnumber': 3, 'dname': 'Research', 'dnumber': 5, 'emp_count': 2, 'avg_salary': 39000.0}
{'pname': 'Reorganization', 'pnumber': 20, 'dname': 'Headquarters', 'dnumber': 1, 'emp_count': 3, 'avg_salary': 46000.0}


<span style=color:blue>Working on groupby/having query from 2015 final    </span>

The query is:

select d.dname, m.fname, m.lname, count(*), avg(e.salary) as avg_salary
from employee e, department d, employee m 
where e.dno = d.dnumber
  and d.mgr_ssn = m.ssn 
group by d.dname, m.fname, m.lname, m.salary
having avg(e.salary) < 50000
order by d.dname

In [52]:
db_company = client.company

pipeline = [
    # 1. Join with employee as emp
    {
        "$lookup": {
            "from": "employee",
            "localField": "dnumber",
            "foreignField": "dno",
            "as": "emp"
        }
    },
    {"$unwind": "$emp"},

    # 2. Join with employee as mgr
    {
        "$lookup": {
            "from": "employee",
            "localField": "mgr_ssn",
            "foreignField": "ssn",
            "as": "mgr"
        }
    }
    ,
    {"$unwind": "$mgr"},

    # 3. Restrict to managers with salary <= 45000
    {
        "$match": {
            "mgr.salary" : {
                "$lte" : 45000
            }
        }
    },

    # 4. Group by dept dname, mgr fname, mgr lname, 
    {
        "$group": {
            "_id": {
                "dname": "$dname",
                "mgr_fname": "$mgr.fname",
                "mgr_lname": "$mgr.lname",
                "mgr_salary": "$mgr.salary"
            },
            "emp_count": {"$sum": 1},
            "avg_salary": {"$avg": "$emp.salary"}
        }
    },

    # 5. Filter: avg_salary of dept is < 50000
    {"$match": {"avg_salary": {"$lt": 50000}}},

    # 6. Sort by dname
    {"$sort": {"_id.dname": 1}},
    
    # 7. Optional: reshape output
    {
        "$project": {
            "_id": 0,
            "dname": "$_id.dname",
            "fname": "$_id.mgr_fname",
            "lname": "$_id.mgr_lname",
            "emp_count": 1,
            "avg_salary": 1
        }
    }
]


result = db_company.department.aggregate(pipeline)

"""
# Print result
for doc in result:
    # print key-value pairs in desired order
    ordered = {
        "dname": doc["dname"],
        "fname": doc["fname"],
        "lname": doc["lname"],
        "emp_count": doc["emp_count"],
        "avg_salary": doc["avg_salary"]
    }
    pprint.pp(ordered)
"""

for doc in result:
    pprint.pp(doc)

{'emp_count': 3,
 'avg_salary': 31000.0,
 'dname': 'Administration',
 'fname': 'Jennifer',
 'lname': 'Wallace'}
{'emp_count': 4,
 'avg_salary': 33250.0,
 'dname': 'Research',
 'fname': 'Franklin',
 'lname': 'Wong'}


<span style=color:blue>Simpler example, grouping just on dname     </span>

In [54]:
db_company = client.company

pipeline = [
    # 1. Join with employee as emp
    {
        "$lookup": {
            "from": "employee",
            "localField": "dnumber",
            "foreignField": "dno",
            "as": "emp"
        }
    },
    {"$unwind": "$emp"},

    # 2. Join with employee as mgr
    {
        "$lookup": {
            "from": "employee",
            "localField": "mgr_ssn",
            "foreignField": "ssn",
            "as": "mgr"
        }
    }
    ,
    {"$unwind": "$mgr"},

    # 3. Restrict to managers with salary <= 45000
    {
        "$match": {
            "mgr.salary" : {
                "$lte" : 45000
            }
        }
    },

    # 4. Group by dept dname, mgr fname, mgr lname, 
    {
        "$group": {
            "_id": "$dname",
            "emp_count": {"$sum": 1},
            "avg_salary": {"$avg": "$emp.salary"}
        }
    },

    # 5. Filter: avg_salary of dept is < 50000
    {"$match": {"avg_salary": {"$lt": 50000}}},

    # 6. Sort by dname
    {"$sort": {"_id.dname": 1}}
]

result = db_company.department.aggregate(pipeline)

for doc in result:
    pprint.pp(doc)

{'_id': 'Research', 'emp_count': 4, 'avg_salary': 33250.0}
{'_id': 'Administration', 'emp_count': 3, 'avg_salary': 31000.0}
